# Cross-Validation with Over/Undersampling
## Joint Sentiment & Intent Classification — Sinhala NLP

**Strategy:**
- Stratified K-Fold on a **composite label** (sentiment × intent) to preserve the joint distribution across all folds.
- Per-fold: sampling is applied to the **training indices only** — the val fold is never touched (prevents data leakage).
- Reports per-fold and aggregate metrics for sentiment, intent, and joint accuracy.

**Sampling modes** (set via `SAMPLING_STRATEGY`):
| Value | Behaviour |
|---|---|
| `"oversample"` | RandomOverSampler — duplicate minority composite-label rows |
| `"undersample"` | RandomUnderSampler — drop majority rows |
| `"both"` | Undersample majority to 3× minority, then oversample minority up |
| `"none"` | No resampling (baseline) |

> **Requires:** `pip install imbalanced-learn`

## 0 — Install Dependencies

In [ ]:
!pip install imbalanced-learn -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1 — Imports

In [ ]:
import os, re, unicodedata, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from tqdm import tqdm

print("All imports OK")

All imports OK


## 2 — Configuration

Edit `SAMPLING_STRATEGY` and `N_FOLDS` here before running.

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
FULL_CSV = "/content/drive/MyDrive/Research Datasets/sinhala_silver_labels.csv"

# ── Model ─────────────────────────────────────────────────────────────────────
BACKBONE     = "xlm-roberta-base"
MAX_LEN      = 128
BATCH_SIZE   = 16 # Reduced batch size to mitigate OutOfMemoryError
DROPOUT      = 0.3

# ── Training ──────────────────────────────────────────────────────────────────
EPOCHS       = 3       # reduced per-fold; early stopping will catch anyway
LR           = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
GRAD_CLIP    = 1.0
ALPHA        = 0.5     # weight between sentiment loss and intent loss
PATIENCE     = 3
SEED         = 42
SENT_SMOOTH  = 0.10
INTENT_SMOOTH = 0.15

# ── Cross-validation & sampling ───────────────────────────────────────────────
N_FOLDS           = 5
SAMPLING_STRATEGY = "oversample"   # "oversample" | "undersample" | "both" | "none"

# ── Labels ────────────────────────────────────────────────────────────────────
SENTIMENT_CLASSES = ["positive", "negative", "neutral", "conflict"]
INTENT_CLASSES    = ["suggestion", "complaint", "inquiry", "feedback", "greeting", "other"]
NUM_SENTIMENTS    = len(SENTIMENT_CLASSES)
NUM_INTENTS       = len(INTENT_CLASSES)

SENTIMENT_MAP = {
    "positive": 0, "Positive": 0,
    "negative": 1, "Negative": 1,
    "neutral":  2, "Neutral":  2,
    "conflict": 3, "Conflict": 3,
}
INTENT_MAP = {
    "inquiry": 0,   "Inquiry": 0,
    "complaint": 1, "Complaint": 1,
    "suggestion": 2,"Suggestion": 2,
    "feedback": 3,  "Feedback": 3,
    "greeting": 4,  "Greeting": 4,
    "other": 5,     "Other": 5,
}

TEXT_COL      = "text"
SENTIMENT_COL = "sentiment_label"
INTENT_COL    = "intent_label"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Sampling strategy: {SAMPLING_STRATEGY}  |  Folds: {N_FOLDS}")

Device: cuda
Sampling strategy: oversample  |  Folds: 5


## 3 — Reproducibility

In [ ]:
def set_seed(s=SEED):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

set_seed()
print(f"Seed set to {SEED}")

Seed set to 42


## 4 — Data Loading & Preprocessing

In [ ]:
def clean_text(text):
    if not isinstance(text, str): return ""
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def load_and_clean(path):
    df = pd.read_csv(path)
    df.rename(columns={"Sentence": "text", "Sentiment": "sentiment_label"}, inplace=True)

    # Warn about any label values not in our maps
    unknown_s = set(df[SENTIMENT_COL].unique()) - set(SENTIMENT_MAP)
    unknown_i = set(df[INTENT_COL].unique())   - set(INTENT_MAP)
    if unknown_s: print("WARNING – unknown sentiment values:", unknown_s)
    if unknown_i: print("WARNING – unknown intent values:",   unknown_i)

    df[SENTIMENT_COL] = df[SENTIMENT_COL].map(SENTIMENT_MAP)
    df[INTENT_COL]    = df[INTENT_COL].map(INTENT_MAP)
    before = len(df)
    df = df.dropna(subset=[SENTIMENT_COL, INTENT_COL])
    df[SENTIMENT_COL] = df[SENTIMENT_COL].astype(int)
    df[INTENT_COL]    = df[INTENT_COL].astype(int)

    df[TEXT_COL] = df[TEXT_COL].apply(clean_text)
    df = df[df[TEXT_COL].str.len() > 0].reset_index(drop=True)
    print(f"Loaded {len(df)} rows (dropped {before - len(df)} unmapped).")
    return df


def composite_label(df):
    """Encode (sentiment, intent) as a single integer for stratified splitting."""
    return df[SENTIMENT_COL] * 10 + df[INTENT_COL]


df = load_and_clean(FULL_CSV)

print("\nSentiment distribution:")
for lbl, cnt in df[SENTIMENT_COL].value_counts().sort_index().items():
    print(f"  [{lbl}] {SENTIMENT_CLASSES[lbl]:<12} {cnt:>5} ({cnt/len(df)*100:.1f}%)")
print("\nIntent distribution:")
for lbl, cnt in df[INTENT_COL].value_counts().sort_index().items():
    print(f"  [{lbl}] {INTENT_CLASSES[lbl]:<12} {cnt:>5} ({cnt/len(df)*100:.1f}%)")

df.head()

Loaded 5881 rows (dropped 0 unmapped).

Sentiment distribution:
  [0] positive       397 (6.8%)
  [1] negative      1534 (26.1%)
  [2] neutral       3888 (66.1%)
  [3] conflict        62 (1.1%)

Intent distribution:
  [0] suggestion    3208 (54.5%)
  [1] complaint      541 (9.2%)
  [2] inquiry        448 (7.6%)
  [3] feedback       785 (13.3%)
  [4] greeting       593 (10.1%)
  [5] other          306 (5.2%)


,text,sentiment_label,intent_label,combined_label,nli_conf,conf_tier,nli_rule_agree,sample_weight
0,Ammage Adarayta❤️Eka Dawasak Madi Neda❤️🙏❤️,1,0,Negative_inquiry,0.3559,rule,True,1.0
1,We need IPL Champions leak data offers ..pleas...,2,0,Neutral_inquiry,0.7913,high,False,1.0
2,chandimal.. uuu thama mulu tem ekama kaaa gaha...,0,0,Positive_inquiry,0.4922,rule,False,1.0
3,Lebsack 49k dammama eka dawasak wath be,1,0,Negative_inquiry,0.8232,rule,True,1.0
4,eth anith kattiya sathiyak withara online inna...,2,5,Neutral_other,0.6918,mid,False,0.5


## 5 — Dataset & DataLoader

In [ ]:
class JointDataset(Dataset):
    def __init__(self, texts, sent_lbls, intent_lbls, tokenizer):
        self.texts       = texts
        self.sent_lbls   = sent_lbls
        self.intent_lbls = intent_lbls
        self.tok         = tokenizer

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tok(str(self.texts[idx]), max_length=MAX_LEN,
                       padding="max_length", truncation=True, return_tensors="pt")
        return {
            "input_ids":       enc["input_ids"].squeeze(0),
            "attention_mask":  enc["attention_mask"].squeeze(0),
            "sentiment_label": torch.tensor(self.sent_lbls[idx],   dtype=torch.long),
            "intent_label":    torch.tensor(self.intent_lbls[idx], dtype=torch.long),
        }


def make_loader(df, tok, shuffle=False):
    ds = JointDataset(df[TEXT_COL].tolist(),
                      df[SENTIMENT_COL].tolist(),
                      df[INTENT_COL].tolist(), tok)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=2, pin_memory=True)


print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BACKBONE)
print(f"Tokenizer ready  (vocab size: {tokenizer.vocab_size:,})")

Loading tokenizer...
Tokenizer ready  (vocab size: 250,002)


## 6 — Model Architecture

In [ ]:
class JointModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(BACKBONE)
        h = self.encoder.config.hidden_size
        # Simpler head for cleaner sentiment labels
        self.sent_head = nn.Sequential(
            nn.Dropout(DROPOUT), nn.Linear(h, NUM_SENTIMENTS))
        # Deeper head for noisier intent labels
        self.intent_head = nn.Sequential(
            nn.Dropout(DROPOUT), nn.Linear(h, 256),
            nn.GELU(), nn.Dropout(DROPOUT), nn.Linear(256, NUM_INTENTS))

    def forward(self, input_ids, attention_mask):
        cls = self.encoder(input_ids=input_ids,
                           attention_mask=attention_mask).last_hidden_state[:, 0, :]
        return self.sent_head(cls), self.intent_head(cls)

    def param_groups(self, lr, wd):
        """Separate param groups: encoder at lr, heads at lr*10; no decay on bias/LN."""
        nd  = ["bias", "LayerNorm.weight"]
        enc = list(self.encoder.named_parameters())
        hd  = (list(self.sent_head.named_parameters()) +
               list(self.intent_head.named_parameters()))
        return [
            {"params": [p for n,p in enc if not any(d in n for d in nd)], "lr": lr,    "weight_decay": wd},
            {"params": [p for n,p in enc if     any(d in n for d in nd)], "lr": lr,    "weight_decay": 0},
            {"params": [p for n,p in hd  if not any(d in n for d in nd)], "lr": lr*10, "weight_decay": wd},
            {"params": [p for n,p in hd  if     any(d in n for d in nd)], "lr": lr*10, "weight_decay": 0},
        ]


# Sanity-check forward pass with a dummy batch
print("Loading backbone for forward-pass check (~1.1 GB on first run)...")
_m = JointModel().to(DEVICE)
_d = torch.zeros(2, MAX_LEN, dtype=torch.long).to(DEVICE)
_k = torch.ones(2, MAX_LEN, dtype=torch.long).to(DEVICE)
_s, _i = _m(_d, _k)
print(f"Forward pass OK  → sentiment {tuple(_s.shape)}  intent {tuple(_i.shape)}")
print(f"Total parameters: {sum(p.numel() for p in _m.parameters()):,}")
del _m, _d, _k, _s, _i

Loading backbone for forward-pass check (~1.1 GB on first run)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Forward pass OK  → sentiment (2, 4)  intent (2, 6)
Total parameters: 278,245,130


## 7 — Loss Function

In [ ]:
# Label-smoothed cross-entropy for each task
sc = nn.CrossEntropyLoss(label_smoothing=SENT_SMOOTH)
ic = nn.CrossEntropyLoss(label_smoothing=INTENT_SMOOTH)

def jloss(sl, il, slbl, ilbl):
    """Weighted sum of sentiment + intent losses (ALPHA controls the balance)."""
    ls = sc(sl, slbl)
    li = ic(il, ilbl)
    return ALPHA * ls + (1 - ALPHA) * li, ls, li

print(f"Joint loss = {ALPHA}×sent_loss + {1-ALPHA}×intent_loss  (label smoothing: sent={SENT_SMOOTH}, intent={INTENT_SMOOTH})")

Joint loss = 0.5×sent_loss + 0.5×intent_loss  (label smoothing: sent=0.1, intent=0.15)


## 8 — Sampling Helpers

Resampling operates on the **composite label** so both task dimensions are balanced together.
Only ever called on the training split — the validation fold is never touched.

In [ ]:
def apply_sampling(df_train, strategy):
    """
    Resample df_train using the composite (sentiment × intent) label.
    Returns a new DataFrame with resampled rows.
    IMPORTANT: Only apply to training data — never to validation.
    """
    if strategy == "none":
        return df_train

    comp = (df_train[SENTIMENT_COL] * 10 + df_train[INTENT_COL]).values.reshape(-1, 1)
    idx  = np.arange(len(df_train)).reshape(-1, 1)

    if strategy == "oversample":
        sampler = RandomOverSampler(random_state=SEED)
        idx_res, _ = sampler.fit_resample(idx, comp)

    elif strategy == "undersample":
        sampler = RandomUnderSampler(random_state=SEED)
        idx_res, _ = sampler.fit_resample(idx, comp)

    elif strategy == "both":
        # Step 1: undersample majority down to 3× minority count
        minority_count  = pd.Series(comp.ravel()).value_counts().min()
        target_majority = min(3 * minority_count,
                              pd.Series(comp.ravel()).value_counts().max())
        sampling_dict_under = {
            cls: min(cnt, target_majority)
            for cls, cnt in pd.Series(comp.ravel()).value_counts().items()
        }
        u_sampler = RandomUnderSampler(sampling_strategy=sampling_dict_under,
                                       random_state=SEED)
        idx_mid, comp_mid = u_sampler.fit_resample(idx, comp)
        # Step 2: oversample minority up to new majority count
        o_sampler = RandomOverSampler(random_state=SEED)
        idx_res, _ = o_sampler.fit_resample(idx_mid, comp_mid)

    else:
        raise ValueError(f"Unknown sampling strategy: '{strategy}'. "
                          "Choose from: oversample | undersample | both | none")

    return df_train.iloc[idx_res.ravel()].reset_index(drop=True)


print("apply_sampling() ready")

apply_sampling() ready


## 9 — Training & Evaluation Loops

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, epoch, n_epochs):
    model.train()
    tot = tsl = til = 0.0
    pbar = tqdm(loader, desc=f"  Train {epoch}/{n_epochs}", leave=False,
                unit="batch", dynamic_ncols=True)
    for i, b in enumerate(pbar, 1):
        ids   = b["input_ids"].to(DEVICE)
        mask  = b["attention_mask"].to(DEVICE)
        s_lbl = b["sentiment_label"].to(DEVICE)
        i_lbl = b["intent_label"].to(DEVICE)
        sl, il = model(ids, mask)
        loss, ls, li = jloss(sl, il, s_lbl, i_lbl)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()
        tot += loss.item(); tsl += ls.item(); til += li.item()
        pbar.set_postfix(loss=f"{tot/i:.4f}", s_loss=f"{tsl/i:.4f}", i_loss=f"{til/i:.4f}")
    n = len(loader)
    return tot/n, tsl/n, til/n


@torch.no_grad()
def eval_epoch(model, loader, epoch, n_epochs):
    model.eval()
    tot = cs = ci = jc = total = 0
    all_sp, all_ip, all_sl, all_il = [], [], [], []
    pbar = tqdm(loader, desc=f"  Val   {epoch}/{n_epochs}", leave=False,
                unit="batch", dynamic_ncols=True)
    for b in pbar:
        ids   = b["input_ids"].to(DEVICE)
        mask  = b["attention_mask"].to(DEVICE)
        s_lbl = b["sentiment_label"].to(DEVICE)
        i_lbl = b["intent_label"].to(DEVICE)
        sl, il = model(ids, mask)
        loss, _, _ = jloss(sl, il, s_lbl, i_lbl)
        tot += loss.item()
        sp = sl.argmax(1); ip = il.argmax(1)
        cs    += (sp == s_lbl).sum().item()
        ci    += (ip == i_lbl).sum().item()
        jc    += ((sp == s_lbl) & (ip == i_lbl)).sum().item()
        total += s_lbl.size(0)
        all_sp.extend(sp.cpu().tolist()); all_ip.extend(ip.cpu().tolist())
        all_sl.extend(s_lbl.cpu().tolist()); all_il.extend(i_lbl.cpu().tolist())
        pbar.set_postfix(loss=f"{tot/max(1, pbar.n):.4f}",
                         s_acc=f"{cs/total:.4f}", i_acc=f"{ci/total:.4f}")
    return {
        "loss":   tot / len(loader),
        "s_acc":  cs / total,
        "i_acc":  ci / total,
        "j_acc":  jc / total,
        "s_f1":   f1_score(all_sl, all_sp, average="macro", zero_division=0),
        "i_f1":   f1_score(all_il, all_ip, average="macro", zero_division=0),
        "all_sp": all_sp, "all_ip": all_ip,
        "all_sl": all_sl, "all_il": all_il,
    }


print("train_epoch() and eval_epoch() ready")

train_epoch() and eval_epoch() ready


## 10 — Cross-Validation Loop

This is the main training loop. For each fold it:
1. Splits data via `StratifiedKFold` on the composite label
2. Applies sampling to the training split only
3. Trains a fresh model with early stopping
4. Records best validation metrics

In [ ]:
import warnings

warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
def run_cv():
    comp_labels = composite_label(df).values
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

    fold_results = []
    fold_pbar = tqdm(enumerate(skf.split(np.zeros(len(df)), comp_labels)),
                     total=N_FOLDS, desc="Folds", unit="fold", dynamic_ncols=True)

    for fold, (train_idx, val_idx) in fold_pbar:
        print(f"\n{'='*60}")
        print(f"  FOLD {fold+1}/{N_FOLDS}")
        print(f"{'='*60}")

        df_train_raw = df.iloc[train_idx].reset_index(drop=True)
        df_val       = df.iloc[val_idx].reset_index(drop=True)

        # ── Class distribution BEFORE sampling ────────────────────────────
        print(f"\n[Fold {fold+1}] Train size before sampling: {len(df_train_raw)}")
        print("  Sentiment distribution (train):")
        for lbl, cnt in df_train_raw[SENTIMENT_COL].value_counts().sort_index().items():
            print(f"    [{lbl}] {SENTIMENT_CLASSES[lbl]:<12} {cnt:>5} ({cnt/len(df_train_raw)*100:.1f}%)")
        print("  Intent distribution (train):")
        for lbl, cnt in df_train_raw[INTENT_COL].value_counts().sort_index().items():
            print(f"    [{lbl}] {INTENT_CLASSES[lbl]:<12} {cnt:>5} ({cnt/len(df_train_raw)*100:.1f}%)")

        # ── Apply sampling (training data ONLY) ───────────────────────────
        df_train = apply_sampling(df_train_raw, SAMPLING_STRATEGY)
        print(f"\n[Fold {fold+1}] Train size after  sampling: {len(df_train)}")
        print("  Sentiment distribution (after sampling):")
        for lbl, cnt in df_train[SENTIMENT_COL].value_counts().sort_index().items():
            print(f"    [{lbl}] {SENTIMENT_CLASSES[lbl]:<12} {cnt:>5} ({cnt/len(df_train)*100:.1f}%)")
        print("  Intent distribution (after sampling):")
        for lbl, cnt in df_train[INTENT_COL].value_counts().sort_index().items():
            print(f"    [{lbl}] {INTENT_CLASSES[lbl]:<12} {cnt:>5} ({cnt/len(df_train)*100:.1f}%)")

        train_loader = make_loader(df_train, tokenizer, shuffle=True)
        val_loader   = make_loader(df_val,   tokenizer)

        # ── Fresh model per fold ──────────────────────────────────────────
        set_seed(SEED + fold)
        model     = JointModel().to(DEVICE)
        optimizer = torch.optim.AdamW(model.param_groups(LR, WEIGHT_DECAY))
        total_steps = len(train_loader) * EPOCHS
        scheduler   = get_linear_schedule_with_warmup(
            optimizer, int(total_steps * WARMUP_RATIO), total_steps)

        best_loss = float("inf"); pat = 0; best_metrics = None

        for epoch in range(1, EPOCHS + 1):
            tl, tsl, til = train_epoch(model, train_loader, optimizer, scheduler, epoch, EPOCHS)
            vm = eval_epoch(model, val_loader, epoch, EPOCHS)
            print(f"  Epoch {epoch}/{EPOCHS}  "
                  f"train_loss={tl:.4f}  val_loss={vm['loss']:.4f}  "
                  f"s_acc={vm['s_acc']:.4f}  i_acc={vm['i_acc']:.4f}  "
                  f"j_acc={vm['j_acc']:.4f}  s_f1={vm['s_f1']:.4f}  i_f1={vm['i_f1']:.4f}")
            if vm["loss"] < best_loss:
                best_loss = vm["loss"]; pat = 0; best_metrics = vm
                torch.save({
                    "fold": fold,
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "val_metrics": vm,
                }, f"/content/drive/MyDrive/Kaggle-datasets/best_model_fold{fold+1}.pt")
                print("    ↳ best checkpoint saved")
            else:
                pat += 1
                if pat >= PATIENCE:
                    print(f"  Early stopping at epoch {epoch}"); break

        fold_results.append(best_metrics)
        fold_pbar.set_postfix(j_acc=f"{best_metrics['j_acc']:.4f}",
                              s_f1=f"{best_metrics['s_f1']:.4f}",
                              i_f1=f"{best_metrics['i_f1']:.4f}")
        print(f"\n[Fold {fold+1}] Best val  → "
              f"j_acc={best_metrics['j_acc']:.4f}  "
              f"s_f1={best_metrics['s_f1']:.4f}  "
              f"i_f1={best_metrics['i_f1']:.4f}")

    return fold_results


fold_results = run_cv()

Folds:   0%|          | 0/5 [00:00<?, ?fold/s]


  FOLD 1/5

[Fold 1] Train size before sampling: 4704
  Sentiment distribution (train):
    [0] positive       317 (6.7%)
    [1] negative      1226 (26.1%)
    [2] neutral       3112 (66.2%)
    [3] conflict        49 (1.0%)
  Intent distribution (train):
    [0] suggestion    2566 (54.5%)
    [1] complaint      433 (9.2%)
    [2] inquiry        358 (7.6%)
    [3] feedback       627 (13.3%)
    [4] greeting       475 (10.1%)
    [5] other          245 (5.2%)

[Fold 1] Train size after  sampling: 45144
  Sentiment distribution (after sampling):
    [0] positive     11286 (25.0%)
    [1] negative     11286 (25.0%)
    [2] neutral      11286 (25.0%)
    [3] conflict     11286 (25.0%)
  Intent distribution (after sampling):
    [0] suggestion    7524 (16.7%)
    [1] complaint     7524 (16.7%)
    [2] inquiry       7524 (16.7%)
    [3] feedback      7524 (16.7%)
    [4] greeting      7524 (16.7%)
    [5] other         7524 (16.7%)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Streaming output truncated to the last 5000 lines.
  Train 1/3: 100%|██████████| 2822/2822 [21:35<00:00,  2.45batch/s, i_loss=0.9575, loss=0.8216, s_loss=0.6858]
                                                                                                              
  Val   1/3:  99%|█████████▊| 73/74 [00:09<00:00,  7.90batch/s, i_acc=0.8029, loss=1.0460, s_acc=0.6661]
                                                                                                        

  Epoch 1/3  train_loss=0.8216  val_loss=1.0319  s_acc=0.6661  i_acc=0.8029  j_acc=0.5344  s_f1=0.5057  i_f1=0.7105
    ↳ best checkpoint saved


Streaming output truncated to the last 5000 lines.
  Train 2/3: 100%|██████████| 2822/2822 [21:36<00:00,  2.45batch/s, i_loss=0.6160, loss=0.5063, s_loss=0.3965]
                                                                                                              
  Val   2/3:  99%|█████████▊| 73/74 [00:09<00:00,  7.87batch/s, i_acc=0.8241, loss=0.9670, s_acc=0.7698]
                                                                                                        

  Epoch 2/3  train_loss=0.5063  val_loss=0.9540  s_acc=0.7698  i_acc=0.8241  j_acc=0.6483  s_f1=0.5435  i_f1=0.7292
    ↳ best checkpoint saved


Streaming output truncated to the last 5000 lines.
  Train 3/3: 100%|██████████| 2822/2822 [21:38<00:00,  2.44batch/s, i_loss=0.5930, loss=0.4802, s_loss=0.3673]
                                                                                                              
Folds:  20%|██        | 1/5 [1:05:34<4:22:18, 3934.66s/fold, i_f1=0.7292, j_acc=0.6483, s_f1=0.5435]

  Epoch 3/3  train_loss=0.4802  val_loss=0.9690  s_acc=0.7715  i_acc=0.8369  j_acc=0.6593  s_f1=0.5127  i_f1=0.7461

[Fold 1] Best val  → j_acc=0.6483  s_f1=0.5435  i_f1=0.7292

  FOLD 2/5

[Fold 2] Train size before sampling: 4705
  Sentiment distribution (train):
    [0] positive       318 (6.8%)
    [1] negative      1227 (26.1%)
    [2] neutral       3110 (66.1%)
    [3] conflict        50 (1.1%)
  Intent distribution (train):
    [0] suggestion    2567 (54.6%)
    [1] complaint      433 (9.2%)
    [2] inquiry        359 (7.6%)
    [3] feedback       628 (13.3%)
    [4] greeting       474 (10.1%)
    [5] other          244 (5.2%)

[Fold 2] Train size after  sampling: 45120
  Sentiment distribution (after sampling):
    [0] positive     11280 (25.0%)
    [1] negative     11280 (25.0%)
    [2] neutral      11280 (25.0%)
    [3] conflict     11280 (25.0%)
  Intent distribution (after sampling):
    [0] suggestion    7520 (16.7%)
    [1] complaint     7520 (16.7%)
    [2] inquiry      

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 11 — Cross-Validation Summary

In [ ]:
print(f"{'='*60}")
print("  CROSS-VALIDATION SUMMARY")
print(f"  Sampling strategy : {SAMPLING_STRATEGY}")
print(f"  Folds             : {N_FOLDS}")
print(f"{'='*60}")

for key, name in [("j_acc","Joint Acc"), ("s_acc","Sent Acc"),
                  ("i_acc","Intent Acc"), ("s_f1","Sent Macro-F1"),
                  ("i_f1","Intent Macro-F1")]:
    vals = [r[key] for r in fold_results]
    print(f"  {name:<18}: {np.mean(vals):.4f} ± {np.std(vals):.4f}  "
          f"(folds: {[round(v,4) for v in vals]})")

## 12 — Detailed Classification Reports (Last Fold)

In [ ]:
last = fold_results[-1]
present_s = sorted(set(last["all_sl"]))
present_i = sorted(set(last["all_il"]))

print("[Last fold] Sentiment classification report:")
print(classification_report(last["all_sl"], last["all_sp"],
                            labels=present_s,
                            target_names=[SENTIMENT_CLASSES[i] for i in present_s],
                            digits=4, zero_division=0))

print("\n[Last fold] Intent classification report:")
print(classification_report(last["all_il"], last["all_ip"],
                            labels=present_i,
                            target_names=[INTENT_CLASSES[i] for i in present_i],
                            digits=4, zero_division=0))

#13 — Load Best Model for Inference

In [ ]:
# Pick the fold with the highest joint accuracy
best_fold_idx = max(range(len(fold_results)), key=lambda i: fold_results[i]["j_acc"])
best_ckpt_path = f"/content/drive/MyDrive/Kaggle-datasets/best_model_fold{best_fold_idx+1}.pt"
print(f"Best fold: {best_fold_idx+1}  →  loading {best_ckpt_path}")

inference_model = JointModel().to(DEVICE)
ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
inference_model.load_state_dict(ckpt["model_state_dict"])
inference_model.eval()
print(f"Loaded checkpoint from fold {ckpt['fold']+1}, epoch {ckpt['epoch']}")
print(f"Val metrics at save: {ckpt['val_metrics']}")


def predict(texts: list[str]) -> list[dict]:
    enc = tokenizer(
        texts,
        max_length=MAX_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
    input_ids      = enc["input_ids"].to(DEVICE)
    attention_mask = enc["attention_mask"].to(DEVICE)

    with torch.no_grad():
        s_logits, i_logits = inference_model(input_ids, attention_mask)

    s_preds = s_logits.argmax(dim=-1).cpu().tolist()
    i_preds = i_logits.argmax(dim=-1).cpu().tolist()

    return [
        {
            "text":      t,
            "sentiment": SENTIMENT_CLASSES[s],
            "intent":    INTENT_CLASSES[i],
        }
        for t, s, i in zip(texts, s_preds, i_preds)
    ]

samples = ["උඹ හරිම හොඳයි", "මට ගෙදර යන්න ඕනි"]
for result in predict(samples):
    print(result)